# AELIONIX BLACKFORGE — Phase 13 Colab Validation

This notebook performs a deterministic, one-click validation of the **Source &
Runtime Correlation Capability Foundation** (Phase 13).

It exercises the full `blackforge.source_runtime` pipeline over deterministic,
independent **declared** (source) and **observed** (runtime) fixture sides for
an `aelionix` cluster/registry/cloud estate:

* **two-independent-sources correlation** — source fixtures and runtime
  fixtures live in separate modules and are never derived from one another;
  correlation joins the two sides, it never authors either fact
* **canonical-identity pairing** — context-mapped attribute translation with
  an opaque identity string (`scope_kind:scope:name`), so the same service in
  a different scope is **never** silently matched
* **deterministic rule engine** — exact equality, set equality, immutable
  digest, presence/absence, and a contradiction-sensitive property set, all
  registered as versioned `CorrelationRule`s with no freeform interpretation
* **evidence integrity** — correlation evidence is `DERIVED` from source and
  runtime evidence (`DERIVED_FROM` links), the source/runtime sides stay
  `OBSERVED`, and credential material is redacted (stable `REDACTED` marker)
  before anything is stored
* **world-model materialization** — every correlated service becomes a
  `SOURCE_COMPONENT` entity with `:source` / `:runtime` siblings and
  `DECLARED_AS` / `OBSERVED_AS` / `CORRESPONDS_TO` / `DIFFERS_FROM` edges;
  a `correlation_result` assertion records the outcome; re-running
  corroborates instead of duplicating
* **guarded pipeline** — request validation, scope / authorization, target
  resolution, **fail-closed capability validation**, mock transport (no real
  cluster, registry, or cloud is ever queried or mutated), evidence
  persistence, world-model materialization, and best-effort memory linking

Every capability is risk **LOW**, mode **CONTROLLED**, and supported on
`ASSET`/`CLOUD`/`APPLICATION` targets. Correlation is **not** vulnerability
detection: mismatches surface as `DISCREPANCY` / `CONTRADICTION`, missing
sides as `UNKNOWN`, never automatic exploitation. No attack-graph vocabulary,
no generic command execution surface, no real network or infrastructure I/O.

> Run all cells top-to-bottom. No GPU, no external services, no credentials.
> The notebook fails loudly on any check.


---

In [ ]:
import sys
import platform

print("Blackforge Phase 13 Colab Validation (Source & Runtime Correlation Capability Foundation)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")

---

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import shutil

# -- Configuration (edit here if fork changes) ---------------------------
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# -----------------------------------------------------------------------

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

---

In [ ]:
import subprocess

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)

---

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet

---

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.models",
    "blackforge.evidence",
    "blackforge.evidence.models",
    "blackforge.evidence.store",
    "blackforge.evidence.repository",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.world_model.query",
    "blackforge.world_model.repository",
    "blackforge.world_model.store",
    "blackforge.source_runtime",
    "blackforge.source_runtime.models",
    "blackforge.source_runtime.identity",
    "blackforge.source_runtime.rules",
    "blackforge.source_runtime.correlation",
    "blackforge.source_runtime.redaction",
    "blackforge.source_runtime.source",
    "blackforge.source_runtime.runtime",
    "blackforge.source_runtime.transport",
    "blackforge.source_runtime.evidence",
    "blackforge.source_runtime.materializer",
    "blackforge.source_runtime.capabilities",
    "blackforge.source_runtime.engine",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Source & runtime module imports: PASS")

---

In [ ]:
import subprocess
import sys
import os

print("Running automated test suite...")
# The LLM/torch-heavy files are excluded: importing the HF provider pulls
# ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes. Those
# tests are validated locally and in the Phase 1 notebook.
# The subprocess runs against pristine defaults: BLACKFORGE_* env overrides
# from the runtime are stripped so the suite behaves exactly like CI and no
# ambient config skews assertions (e.g. log level / DB path).
_test_env = {k: v for k, v in os.environ.items() if not k.startswith("BLACKFORGE_")}
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR), env=_test_env,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")

---

In [ ]:
import os
from pathlib import Path

DBROOT = Path("data/phase13_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
for key in ("config_loaded", "mission_manager_ready", "capability_registry_ready",
            "memory_ready", "evidence_store_ready", "evidence_memory_link_ready",
            "world_model_ready", "recon_ready", "webapi_ready", "auth_ready",
            "business_logic_ready", "network_ready", "identity_ready",
            "authorization_ready", "model_router_ready", "cloud_ready",
            "container_ready"):
    assert verification[key], f"{key} must be True"
assert verification["source_runtime_ready"] is True, "source_runtime_ready must be True"
assert len(verification) == 20, len(verification)
assert len(app.capability_registry.list_capabilities()) == 105

BOOTSTRAP_OK = app.healthy() and bool(verification["source_runtime_ready"])

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (source_runtime_ready, 105 registered capabilities): PASS")

---

In [ ]:
from blackforge.source_runtime import (
    SOURCE_RUNTIME_CAPABILITY_IDS,
    CorrelationResult,
    SourceRuntimeMode,
    SourceRuntimeRequest,
    build_source_runtime_meta,
)
from blackforge.scope.models import TargetScope, Target
from blackforge.core.types import RiskLevel, TargetType

MID = "mission_phase13_sr"

CLOUD = "cloud"
CLUSTER = "cluster"
IMAGE = "image"
PROD_PAYMENTS = "aelionix-prod/payments"
STAGING_PAYMENTS = "aelionix-staging/payments"
PROD_MARKETING = "aelionix-prod/marketing"
REGISTRY = "registry.aelionix.io"

_ALL_TARGETS = [CLOUD, CLUSTER, IMAGE, PROD_PAYMENTS, STAGING_PAYMENTS,
                PROD_MARKETING, REGISTRY]

scope = TargetScope(
    mission_id=MID,
    allowed_targets=[
        Target(value=t, target_type=tt)
        for t, tt in (
            (CLOUD, TargetType.CLOUD),
            (CLUSTER, TargetType.APPLICATION),
            (IMAGE, TargetType.ASSET),
            (PROD_PAYMENTS, TargetType.APPLICATION),
            (STAGING_PAYMENTS, TargetType.APPLICATION),
            (PROD_MARKETING, TargetType.APPLICATION),
            (REGISTRY, TargetType.ASSET),
        )
    ],
    max_risk_level=RiskLevel.HIGH,
)
req = SourceRuntimeRequest(
    mission_id=MID,
    session_id="ses_phase13_sr",
    scope=scope,
    mode=SourceRuntimeMode.CONTROLLED,
    max_observations=500,
    timeout_seconds=30.0,
    max_comparisons=1000,
)

engine = app.source_runtime_engine
assert engine is not None and len(engine.capabilities) == 10
ids_seen = [c.capability_id for c in engine.capabilities]
assert set(ids_seen) == set(SOURCE_RUNTIME_CAPABILITY_IDS), (
    ids_seen, SOURCE_RUNTIME_CAPABILITY_IDS)
print("Registered source/runtime capabilities:", ", ".join(ids_seen))

for c in engine.capabilities:
    meta = c.meta()
    risk = meta.risk_level.value
    mode = meta.mode
    assert risk == "low", c.capability_id
    assert mode == "controlled", c.capability_id
    assert meta.supported_target_types, c.capability_id
    assert set(t.value for t in meta.supported_target_types) <= {"asset", "cloud", "application"}
    print(
        f"  {str(meta.id):<48} risk={risk:<7} mode={mode:<9} "
        f"targets={[t.value for t in meta.supported_target_types]}"
    )

# Every correlation capability compares the full result vocabulary.
for m in build_source_runtime_meta():
    assert CorrelationResult.MATCH in m.compares
    assert CorrelationResult.DISCREPANCY in m.compares
    assert CorrelationResult.CONTRADICTION in m.compares

CAPS_OK = True
print("Capability surface (10 low-risk controlled correlations, typed results): PASS")

---

In [ ]:
from blackforge.source_runtime import (
    SourceRuntimeStatus,
    CorrelationResult,
)
from blackforge.evidence.models import (
    EvidenceRelation,
    EvidenceStatus,
    EvidenceType,
)
from blackforge.core.types import Confidence
from blackforge.world_model.query import RelationshipQuery, WorldQuery
from blackforge.world_model.models import EntityType, RelationshipType

CLOUD_CAP = "source_runtime.cloud_resource_correlation"
CONTAINER_CAP = "source_runtime.container_configuration_correlation"
IMAGE_CAP = "source_runtime.image_runtime_correlation"

# -- Cloud umbrella: CONTRADICTION (backup) + DISCREPANCY (version) --------
r_cloud = engine.run(CLOUD_CAP, req, target=CLOUD)
assert r_cloud.status == SourceRuntimeStatus.PARTIAL
assert len(r_cloud.outcomes) == 1
o_cloud = r_cloud.outcomes[0]
assert o_cloud.result == CorrelationResult.CONTRADICTION
assert o_cloud.identity.identity == (
    "cloud:aws:acct-112233445566:us-east-1:payments-database")
contra = [c for c in o_cloud.comparisons if c.result == CorrelationResult.CONTRADICTION]
disc = [c for c in o_cloud.comparisons if c.result == CorrelationResult.DISCREPANCY]
assert len(contra) == 1 and contra[0].property == "backup_enabled"
assert contra[0].source_value == "true" and contra[0].runtime_value == "false"
assert len(disc) == 1 and disc[0].property == "version"
assert disc[0].source_value == "16" and disc[0].runtime_value == "15"
print("Cloud umbrella: CONTRADICTION(backup_enabled) + DISCREPANCY(version) -> PARTIAL: PASS")

# -- Cluster umbrella: MATCH + MATCH + DISCREPANCY + UNKNOWN ---------------
r_cluster = engine.run(CONTAINER_CAP, req, target=CLUSTER)
assert r_cluster.status == SourceRuntimeStatus.PARTIAL
assert len(r_cluster.outcomes) == 4
by_id = {o.identity.identity: o for o in r_cluster.outcomes}
assert by_id["cluster:aelionix-prod:payments:payment-gateway"].result == CorrelationResult.MATCH
assert by_id["cluster:aelionix-staging:payments:payment-gateway"].result == CorrelationResult.MATCH
assert by_id["cluster:aelionix-prod:marketing:marketing-site"].result == CorrelationResult.DISCREPANCY
assert by_id["cluster:aelionix-prod:payments-prod:payment-gateway"].result == CorrelationResult.UNKNOWN
print("Cluster umbrella: 2 MATCH + 1 DISCREPANCY + 1 UNKNOWN -> PARTIAL: PASS")

# -- Scoped targets --------------------------------------------------------
r_prod = engine.run(CONTAINER_CAP, req, target=PROD_PAYMENTS)
assert r_prod.status == SourceRuntimeStatus.SUCCESS
assert r_prod.outcomes[0].result == CorrelationResult.MATCH
assert r_prod.outcomes[0].identity.identity == (
    "cluster:aelionix-prod:payments:payment-gateway")
print("Prod payments scoped: SUCCESS MATCH: PASS")

r_staging = engine.run(CONTAINER_CAP, req, target=STAGING_PAYMENTS)
assert r_staging.status == SourceRuntimeStatus.SUCCESS
assert r_staging.outcomes[0].result == CorrelationResult.MATCH
print("Staging payments scoped: SUCCESS MATCH: PASS")

r_marketing = engine.run(CONTAINER_CAP, req, target=PROD_MARKETING)
assert r_marketing.status == SourceRuntimeStatus.PARTIAL
assert r_marketing.outcomes[0].result == CorrelationResult.DISCREPANCY
replicas = [c for c in r_marketing.outcomes[0].comparisons
            if c.property == "replicas"]
assert len(replicas) == 1
assert replicas[0].source_value == "3" and replicas[0].runtime_value == "5"
print("Marketing scoped: PARTIAL DISCREPANCY on replicas (3 vs 5): PASS")

r_image = engine.run(IMAGE_CAP, req, target=REGISTRY)
assert r_image.status == SourceRuntimeStatus.SUCCESS
assert r_image.outcomes[0].result == CorrelationResult.MATCH
props = {c.property: c.result for c in r_image.outcomes[0].comparisons}
assert props["digest"] == CorrelationResult.MATCH
assert props["tag"] == CorrelationResult.MATCH
print("Image scoped (registry.aelionix.io): SUCCESS MATCH on digest/tag: PASS")

# -- Three evidence rows per outcome: correlation DERIVED from both sides ---
corr_id, src_id, rnt_id = r_cloud.evidence_ids
rows = {e.id: e for e in app.evidence_store.list(limit=10000)}
corr_ev = rows[corr_id]
src_ev = rows[src_id]
rnt_ev = rows[rnt_id]
assert corr_ev.evidence_type == EvidenceType.VALIDATION_RESULT
assert corr_ev.status == EvidenceStatus.INFERRED
assert src_ev.evidence_type == EvidenceType.SOURCE_ANALYSIS
assert src_ev.status == EvidenceStatus.OBSERVED
assert rnt_ev.evidence_type == EvidenceType.OBSERVATION
assert rnt_ev.status == EvidenceStatus.OBSERVED
print("Evidence rows: correlation(INFERRED validation) derived from "
      "source(OBSERVED) + runtime(OBSERVED): PASS")

rels = app.evidence_store.get_relationships(corr_id)
derived = [x for x in rels if x.relation_type == EvidenceRelation.DERIVED_FROM]
assert len(derived) >= 2
targets = {str(x.target_id) for x in derived}
assert str(src_id) in targets and str(rnt_id) in targets
print("DERIVED_FROM links from correlation evidence to source+runtime: PASS")

# -- Evidence dedup: re-running the same scenario reuses the same rows ------
r_cloud2 = engine.run(CLOUD_CAP, req, target=CLOUD)
assert r_cloud2.evidence_ids == r_cloud.evidence_ids
print("Evidence dedup: re-run produced the identical evidence ids: PASS")

# -- World model -------------------------------------------------------------
wm = app.world_model
ents = wm.list_entities(WorldQuery(mission_id=MID, entity_type=EntityType.SOURCE_COMPONENT))
main_name = "cloud:aws:acct-112233445566:us-east-1:payments-database"
names = {e.name for e in ents}
assert main_name in names
assert f"{main_name}:source" in names
assert f"{main_name}:runtime" in names
print("World model: SOURCE_COMPONENT entity + :source / :runtime siblings: PASS")

rels = wm.list_relationships(RelationshipQuery(mission_id=MID))
rel_types = {r.relationship_type.value for r in rels}
assert {"declared_as", "observed_as", "corresponds_to", "differs_from"} <= rel_types
offensive = {"exploits", "can_compromise", "leads_to", "enables",
             "privilege_escalation_path"}
assert rel_types & offensive == set(), rel_types & offensive
print("World edges: DECLARED_AS / OBSERVED_AS / CORRESPONDS_TO / DIFFERS_FROM; "
      "no attack-graph types: PASS")

main_ent = next(e for e in ents if e.name == main_name)
assertions = wm.list_assertions(main_ent.id)
vals = {a.property_key: a.property_value for a in assertions}
assert vals["correlation_result"] == "contradiction"
print("correlation_result assertion materialized on the main entity: PASS")

# -- Result aggregation: a contradiction outranks a discrepancy -------------
from blackforge.source_runtime.correlation import _aggregate_result
assert _aggregate_result([CorrelationResult.MATCH,
                          CorrelationResult.DISCREPANCY,
                          CorrelationResult.CONTRADICTION]) == CorrelationResult.CONTRADICTION
assert _aggregate_result([CorrelationResult.MATCH,
                          CorrelationResult.DISCREPANCY]) == CorrelationResult.DISCREPANCY
assert _aggregate_result([CorrelationResult.MATCH,
                          CorrelationResult.UNKNOWN]) == CorrelationResult.UNKNOWN
PIPELINE_OK = True
print("Result aggregation (CONTRADICTION > DISCREPANCY > UNKNOWN/MATCH): PASS")

---

In [ ]:
from blackforge.source_runtime import (
    CorrelationEvaluator,
    CorrelationResult as CR,
    SourceValue,
    Side,
    build_correlation_identity,
    correlation_confidence,
    credential_value_redacted,
    default_property_resolver,
    default_rule_registry,
    redact_source_runtime_raw,
    same_correlation_identity,
)
from blackforge.core.types import Confidence

# -- Canonical identity ------------------------------------------------
ident = build_correlation_identity("cluster", ["prod", "ns"], "web")
assert ident.identity == "cluster:prod:ns:web", ident.identity
assert same_correlation_identity(
    ident, build_correlation_identity("cluster", ["prod", "ns"], "web")) is True
assert build_correlation_identity("cluster", ["prod", "a"], "svc").identity != (
    build_correlation_identity("cluster", ["staging", "a"], "svc").identity)
print("Canonical identity (<scope_kind>:<scope>:<name>) differs across scope: PASS")

# -- Deterministic rules -------------------------------------------------
_eval = CorrelationEvaluator(default_rule_registry())
same_scope = {"scope_kind": "cluster", "scope_parts": ["prod", "a"], "name": "svc"}
o_match = _eval.compare_records(
    source_record={"identity": dict(same_scope), "properties": {"x": "1"}},
    runtime_record={"identity": dict(same_scope), "properties": {"x": "1"}},
    source_conf=Confidence.MEDIUM, runtime_conf=Confidence.MEDIUM,
    capability_id="cap")
assert o_match.result == CR.MATCH, o_match.result

diff_scope = {"scope_kind": "cluster", "scope_parts": ["staging", "a"], "name": "svc"}
o_nc = _eval.compare_records(
    source_record={"identity": dict(same_scope), "properties": {"x": "1"}},
    runtime_record={"identity": dict(diff_scope), "properties": {"x": "1"}},
    source_conf=Confidence.MEDIUM, runtime_conf=Confidence.MEDIUM,
    capability_id="cap")
assert o_nc is not None and o_nc.result == CR.NOT_COMPARABLE, o_nc
print("Rule engine: equal props MATCH; same name different scope NEVER matches "
      "(NOT_COMPARABLE): PASS")

o_contra = _eval.compare_records(
    source_record={"identity": dict(same_scope), "properties": {"tls_enabled": "true"}},
    runtime_record={"identity": dict(same_scope), "properties": {"tls_enabled": "false"}},
    source_conf=Confidence.MEDIUM, runtime_conf=Confidence.MEDIUM,
    capability_id="cap")
contra = [c for c in o_contra.comparisons if c.result == CR.CONTRADICTION]
assert len(contra) == 1 and contra[0].rule.rule_id == "sr_contradiction"
print("Contradiction-sensitive property routed to sr_contradiction: PASS")

# -- Confidence = weaker of the two independent sides --------------------
assert correlation_confidence(Confidence.LOW, Confidence.HIGH) == Confidence.LOW
assert correlation_confidence(Confidence.HIGH, Confidence.HIGH) == Confidence.HIGH
print("Correlation confidence = weaker side (never inflated by dedup): PASS")

# -- Redaction at the boundary ------------------------------------------
import json
demo_raw = json.dumps({
    "scope_kind": "cluster",
    "registry_token": "demo-registry-token-0000",
    "service_account_token": "demo-sa-token-0000",
    "kubeconfig_password": "demo-kubeconfig-password-0000",
    "labels": ["api", "public"],
})
clean = json.loads(redact_source_runtime_raw(demo_raw))
assert clean["registry_token"] == "REDACTED"
assert clean["service_account_token"] == "REDACTED"
assert clean["kubeconfig_password"] == "REDACTED"
assert clean["labels"] == ["api", "public"]
assert "demo-" not in redact_source_runtime_raw(demo_raw)
print("Redaction (credential-like fields -> stable REDACTED marker): PASS")

# -- Mission isolation ---------------------------------------------------
MID2 = "mission_phase13_sr_other"
scope2 = TargetScope(
    mission_id=MID2,
    allowed_targets=[Target(value=PROD_PAYMENTS, target_type=TargetType.APPLICATION)],
    max_risk_level=RiskLevel.HIGH,
)
req2 = SourceRuntimeRequest(
    mission_id=MID2, session_id="ses_phase13_sr_2", scope=scope2,
    mode=SourceRuntimeMode.CONTROLLED, max_observations=500,
    timeout_seconds=30.0, max_comparisons=1000,
)
r2 = engine.run(CONTAINER_CAP, req2, target=PROD_PAYMENTS)
assert r2.outcomes[0].result == CR.MATCH
other_ids = {str(x) for x in r2.evidence_ids}
assert other_ids.isdisjoint({str(x) for x in r_prod.evidence_ids})
assert app.evidence_store.count(MID2) == len(r2.evidence_ids)
assert wm.count_entities(MID2, entity_type=EntityType.SOURCE_COMPONENT) == 3
print("Mission isolation: second mission produced its own evidence/world rows: PASS")

# -- Idempotency ---------------------------------------------------------
before = app.evidence_store.count(MID)
r_again = engine.run(CONTAINER_CAP, req, target=CLUSTER)
after = app.evidence_store.count(MID)
assert after == before, (before, after)
assert r_again.status == SourceRuntimeStatus.PARTIAL
print("Idempotency: a repeated full-pipeline run reused rows (no growth): PASS")

---

In [ ]:
from blackforge.core.errors import (
    AuthorizationError,
    SourceRuntimeExecutionError,
)
from blackforge.source_runtime import (
    MockSourceRuntimeTransport,
    SourceRuntimeEngine,
    SourceRuntimeStatus,
)

# 1) Target outside the scope is denied BEFORE any transport runs.
denied_out = True
try:
    engine.run(CONTAINER_CAP, req, target="ocs/sneaky-payments")
    denied_out = False
except AuthorizationError:
    pass
assert denied_out, "out-of-scope target ocs/sneaky-payments must be denied"
print("Out-of-scope target denied before transport execution: PASS")

# 2) Unknown capability returns a structured status (fail-closed).
unknown = engine.run("source_runtime.does_not_exist", req, target=CLOUD)
assert unknown.status == SourceRuntimeStatus.UNKNOWN_CAPABILITY
assert "unknown" in unknown.error.lower()
assert len(unknown.outcomes) == 0
print("Unknown capability rejected (UNKNOWN_CAPABILITY, no outcomes): PASS")

# 3) Transport error documents propagate to SourceRuntimeExecutionError.
raw_missing = MockSourceRuntimeTransport().execute(
    "source_runtime.container_configuration_correlation",
    "missing-ns/missing-app")
import json as _json
assert _json.loads(raw_missing)["error"]["kind"] == "no_fixture_records"
try:
    SourceRuntimeEngine()._parse_document(raw_missing, "missing-ns/missing-app")
    raise AssertionError("must raise")
except SourceRuntimeExecutionError as exc:
    assert "no_fixture_records" in str(exc)
print("Transport error document (no_fixture_records) propagates: PASS")

# 4) Correlation does NOT invent facts: runtime-only service -> UNKNOWN.
payments_prod = [o for o in r_cluster.outcomes
                 if "payments-prod" in o.identity.identity]
assert len(payments_prod) == 1
assert payments_prod[0].result == CorrelationResult.UNKNOWN
comparisons = payments_prod[0].comparisons
assert comparisons, "no comparisons"
assert "missing source evidence" in comparisons[0].note
print("Runtime-only service surfaces UNKNOWN (missing source evidence, "
      "never fabricated as MATCH): PASS")

# 5) No generic shell executor surface in the correlator module.
import os as _os
for root, _dirs, files in _os.walk("blackforge/source_runtime"):
    for name in files:
        if not name.endswith(".py"):
            continue
        with open(_os.path.join(root, name), encoding="utf-8") as fh:
            text = fh.read()
        for banned in ("os.system", "subprocess", "socket", "requests.",
                       "urllib.request", "http.client", "eval(", "exec(",
                       "pickle"):
            assert banned not in text, (name, banned)
print("No generic shell executor / network I/O surface in source_runtime: PASS")

---

In [ ]:
from blackforge.evidence.repository import SQLiteEvidenceRepository
from blackforge.evidence.store import EvidenceStore
from blackforge.world_model.repository import SQLiteWorldRepository
from blackforge.world_model.store import WorldModelStore

# Fresh connections over the same SQLite files prove restart persistence.
fresh_ev = EvidenceStore(SQLiteEvidenceRepository(str(DBROOT / "evidence.db")))
fresh_wm = WorldModelStore(SQLiteWorldRepository(str(DBROOT / "world_model.db")))

persisted_ev = fresh_ev.count(MID) == app.evidence_store.count(MID)
persisted_wm = fresh_wm.count_entities(MID) == wm.count_entities(MID)
rel_count = len(fresh_wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000)))
assert persisted_ev and persisted_wm and rel_count > 0
assert fresh_ev.count(MID) > 0
assert fresh_wm.count_entities(MID) > 0
PERSIST_OK = persisted_ev and persisted_wm

for store in (fresh_ev, fresh_wm):
    store.close()
try:
    app.evidence_store.close()
except Exception:
    pass
try:
    app.world_model.close()
except Exception:
    pass
print("Restart persistence (fresh connections on same DB files): PASS")
print("Backends closed. Validation summary below.")

---

In [ ]:
results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "source_runtime" / "engine.py").exists(),
    "phase13_modules": bool(
        (REPO_DIR / "blackforge" / "source_runtime" / "capabilities.py").exists()
        and (REPO_DIR / "blackforge" / "source_runtime" / "evidence.py").exists()
        and (REPO_DIR / "blackforge" / "source_runtime" / "materializer.py").exists()
        and (REPO_DIR / "blackforge" / "source_runtime" / "redaction.py").exists()
        and (REPO_DIR / "blackforge" / "source_runtime" / "correlation.py").exists()
        and (REPO_DIR / "blackforge" / "source_runtime" / "identity.py").exists()
        and (REPO_DIR / "blackforge" / "source_runtime" / "rules.py").exists()
        and (REPO_DIR / "blackforge" / "source_runtime" / "source.py").exists()
        and (REPO_DIR / "blackforge" / "source_runtime" / "runtime.py").exists()
        and (REPO_DIR / "blackforge" / "source_runtime" / "transport.py").exists()
    ),
    "imports": len(_import_failures) == 0,
    "bootstrap_source_runtime_ready": BOOTSTRAP_OK,
    "capability_surface": CAPS_OK,
    "pipeline_evidence": PIPELINE_OK,
    "world_materialized": True,
    "no_attack_graph": not bool(rel_types & offensive),
    "canonical_paired": True,
    "not_comparable_guard": True,
    "confidence_weaker_side": True,
    "authorization_scoped": denied_out,
    "unknown_capability_rejected": True,
    "redaction_boundary": True,
    "mission_isolation": True,
    "idempotent_runs": bool(after == before),
    "restart_persistence": PERSIST_OK,
}

# The pytest cell aborts the run on failure, so reaching this cell proves it passed.
pytest_passed = True
install_ok = len(_import_failures) == 0

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback always works; this notebook needs no GPU
results["Installation"] = install_ok
results["Imports"] = install_ok
results["Automated tests"] = pytest_passed
results["Bootstrap"] = phase_checks["bootstrap_source_runtime_ready"]
results["Phase-specific tests"] = all(phase_checks.values())
results["Security checks"] = (
    phase_checks["authorization_scoped"]
    and phase_checks["unknown_capability_rejected"]
    and phase_checks["redaction_boundary"]
    and phase_checks["no_attack_graph"]
    and phase_checks["not_comparable_guard"]
    and phase_checks["idempotent_runs"]
)

print()
print("=" * 60)
print("PHASE 13 COLAB VALIDATION SUMMARY")
print("=" * 60)
for name, ok in results.items():
    symbol = "PASS" if ok else "FAIL"
    print(f"  [{symbol}] {name}")

_all_ok = all(results.values()) and all(phase_checks.values())
assert _all_ok, "One or more validation checks failed"

print()
print("LOCAL VALIDATION: SUCCESS")
print()
print("Note: this notebook validates the commit checked out into /content/blackforge.")

---